In [42]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from tabulate import tabulate
import numpy as np
import pyfixest as pf

In [43]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"
train_stations_in = "../../../data/interim/train_stations.geojson"

did_out = "../../../reports/replication/did_simple.md"

In [44]:
train_panel = pd.read_parquet(train_panel_in)
bike_panel = pd.read_parquet(bike_panel_in)
uber_panel = pd.read_parquet(uber_panel_in)
train_stations = gpd.read_file(train_stations_in)

# Preprocessing

In [45]:
# The DNC was only 4 days, not even friday. So weekends are uninformative.
# TODO: undo this for the placebo test? or the DNC1 or DNC2 robustness test?

def drop_weekends(df):
    df = df[(0 <= df['dotw']) & (df['dotw'] <= 3)]
    return df

train_panel = train_panel.pipe(drop_weekends)
bike_panel = bike_panel.pipe(drop_weekends)
uber_panel = uber_panel.pipe(drop_weekends)

In [ ]:
def drop_damen(df):
    damen_mask = (train_stations['line'] == 'green') & (train_stations['station_name'] == 'Damen')
    damen_id = train_stations[damen_mask]['map_id'].iloc[0]
    damen_mask = (df['id'] == damen_id)
    df = df[~damen_mask]
    return df

# DiffNDiff

In [98]:
# def model_transit_did(df):
#     df = df.sort_values('date')
#     unit_obs = df.groupby('id').size()
#     df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
#     formula = f"""logrides ~ UCMP : DNC + id + C(time)"""
#     did_model = sm.WLS.from_formula(formula, df, weights=df['weight']).fit(cov_type='cluster',cov_kwds={'groups': df['id']})
#     return did_model

def model_transit_did(df):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    formula = f"""logrides ~ treat | id + time"""
    did_model = pf.feols(formula, df, weights='weight', vcov={"CRV1": "id + time"})
    return did_model

def model_combined_did(df):
    df = df.sort_values('date')
    unit_obs = df.groupby('id').size()
    df['weight'] = df['id'].map(unit_obs).apply(lambda x: 1/x)
    formula = f"""logrides ~ treat : i(transit, ref='uber') + treat | id + time"""
    did_model = pf.feols(formula, df, weights='weight', vcov={"CRV1": "id + time"})
    return did_model

In [99]:
bike_did_model = model_transit_did(bike_panel)
bike_did_model.summary()

###

Estimation:  OLS
Dep. var.: logrides, Fixed effects: id + time
sample: None = all
Inference:  CRV1
Observations:  46024

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treat         |      0.260 |        0.047 |     5.484 |      0.000 |  0.165 |   0.355 |
---
RMSE: 0.057 R2: 0.953 R2 Within: 0.001 


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 123 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: divide by zero encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: overflow encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: invalid value encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:589: RuntimeWarning: divide by zero encountered in matmul
  s

In [100]:
train_did_model = model_transit_did(train_panel)
train_did_model.summary()

###

Estimation:  OLS
Dep. var.: logrides, Fixed effects: id + time
sample: None = all
Inference:  CRV1
Observations:  7912

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treat         |      0.424 |        0.116 |     3.663 |      0.001 |  0.192 |   0.657 |
---
RMSE: 0.016 R2: 0.98 R2 Within: 0.048 


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: divide by zero encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: overflow encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: invalid value encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:589: RuntimeWarning: divide by zero encountered in matmul
  self._tZX = self._Z.T @ self._X
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:589: RuntimeWarning: overflow encountered in matmul
  self._tZX =

In [101]:
uber_did_model = model_transit_did(uber_panel)
uber_did_model.summary()

###

Estimation:  OLS
Dep. var.: logrides, Fixed effects: id + time
sample: None = all
Inference:  CRV1
Observations:  56090

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treat         |      0.231 |        0.082 |     2.815 |      0.007 |  0.067 |   0.396 |
---
RMSE: 0.053 R2: 0.96 R2 Within: 0.001 


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 28 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: divide by zero encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: overflow encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:2489: RuntimeWarning: invalid value encountered in matmul
  tXX = np.ascontiguousarray(X.T @ X, dtype=np.float64)
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:589: RuntimeWarning: divide by zero encountered in matmul
  se

In [102]:
pf.etable([uber_did_model, train_did_model, bike_did_model])

GT(_tbl_data=  __index_level_0__ __index_level_1__                     0  \
0              coef             treat  0.231** <br> (0.082)   
1                fe                id                     -   
2                fe              time                     -   
3             stats      Observations                56,090   
4             stats                R²                  0.96   

                       1                      2  
0  0.424*** <br> (0.116)  0.260*** <br> (0.047)  
1                      -                      -  
2                      -                      -  
3                  7,912                 46,024  
4                   0.98                  0.953  , _body=<great_tables._gt_data.Body object at 0x121635030>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x137a029e0>, _spanners=Spanners([SpannerInfo(spanner_id='logrides', spanner_level=1, spanner_label=Html(text='logrides'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1216374c0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x121634eb0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x121636bf0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x127271b70>, <great_tables._gt_data.FormatInfo object at 0x1272723b0>, <great_tables._gt_data.FormatInfo object at 0x127271a80>, <great_tables._gt_data.FormatInfo object at 0x127271de0>, <great_tables._gt_data.FormatInfo object at 0x127270670>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_

In [97]:
combined = pd.concat([train_panel, bike_panel, uber_panel],
                     keys=['train', 'bike', 'uber'], names=['transit']).reset_index('transit')
assert not any(combined.groupby('id')['transit'].nunique() > 1)
combined_model = model_combined_did(combined)
combined_model.summary()


###

Estimation:  OLS
Dep. var.: logrides, Fixed effects: id + time
sample: None = all
Inference:  CRV1
Observations:  110026

| Coefficient          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| treat                |      0.214 |        0.087 |     2.457 |      0.017 |  0.039 |   0.388 |
| treat:transit::bike  |      0.071 |        0.106 |     0.672 |      0.505 | -0.141 |   0.282 |
| treat:transit::train |      0.126 |        0.162 |     0.781 |      0.438 | -0.198 |   0.451 |
---
RMSE: 0.054 R2: 0.972 R2 Within: 0.001 


/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/formula/model_matrix.py:151: UserWarning: 151 singleton fixed effect(s) dropped from the model.
  warnings.warn(
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:594: RuntimeWarning: divide by zero encountered in matmul
  self._u_hat = self._Y.flatten() - (self._X @ self._beta_hat).flatten()
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:594: RuntimeWarning: overflow encountered in matmul
  self._u_hat = self._Y.flatten() - (self._X @ self._beta_hat).flatten()
/Users/eric/Repos/DNCTransit/.venv/lib/python3.10/site-packages/pyfixest/estimation/models/feols_.py:594: RuntimeWarning: invalid value encountered in matmul
  self._u_hat = self._Y.flatten() - (self._X @ self._beta_hat).flatten()


In [103]:
headers = ["DiD (Uber)", "DiD (Train)", "DiD (Bike)"]

did_summary = summary_col([uber_did_model, train_did_model, bike_did_model], 
            model_names=headers, 
            info_dict={"N":lambda x:(x.nobs)},
            stars=True,
            regressor_order=["UCMP:DNC", "id", "C(time)"],
            drop_omitted=True)

did_summary.tables[0].index = (did_summary.tables[0].index
                               .str.replace('DNC','During DNC')
                               .str.replace('UCMP','Near DNC')
                               .str.replace('C(time)','time fe')
                               .str.replace('id','unit fe'))

did_table = tabulate(did_summary.tables[0],
                   headers=[""] + headers,
                   showindex=True,
                   tablefmt='github')
print(did_table)
with open(did_out,"w") as f:
    f.write(did_table)

# !cp {did_out} ../../../../eric-mc2-cv/static/uploads/

AttributeError: 'Feols' object has no attribute 'params'

## DiD Impact

In [35]:
beta3 = did_summary.tables[0].xs('Near DNC:During DNC').str.replace('*','').astype(float)
main_pct = np.exp(beta3) - 1
pct3 = main_pct.apply("{:.1%}".format)
pct3

DiD (Uber)     826.3%
DiD (Train)      2.3%
DiD (Bike)      29.6%
Name: Near DNC:During DNC, dtype: object